# Python `str.zfill()`: Master Sign-Aware Leading Zero Padding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/plus2net/Python-basics/blob/main/python_string_zfill_practice.ipynb)

Welcome to the interactive companion notebook for the **[plus2net Python str.zfill() Tutorial](https://www.plus2net.com/python/string-zfill.php)**.

### Learning Objectives:
1. Pad numeric strings with leading zeros to enforce fixed-width schemas.
2. Master sign-aware zero insertion after leading `+` and `-` characters.
3. Understand why generic `rjust(width, '0')` corrupts signed numbers while `zfill()` preserves them.
4. Build real-world formatters for enterprise badges, ISO timestamps, and financial deltas.
5. Validate implementations with automated self-checking unit assertions.

## 1. Basic Zero Padding & Width Mechanics

`str.zfill(width)` pads ASCII `'0'` digits to the left until the total string length matches `width`.
If the string is already equal to or longer than `width`, Python returns the string unaltered without truncation.

In [ ]:
record_id = "84"
padded_id = record_id.zfill(6)

print("Original: ", repr(record_id))
print("Padded:   ", repr(padded_id))
assert padded_id == "000084"

# Width smaller than length: never truncates
long_id = "987654321"
assert long_id.zfill(4) == "987654321"

# Empty string
assert "".zfill(4) == "0000"

print("\nAll basic width mechanics assertions passed!")


## 2. Sign Awareness: Preserving `+` and `-` Prefixes

If a string begins with `+` or `-`, `zfill()` inserts padding zeros **after** the sign character.

In [ ]:
pos = "+42"
neg = "-42"

print("Positive padded:", pos.zfill(6))
print("Negative padded:", neg.zfill(6))

assert pos.zfill(6) == "+00042"
assert neg.zfill(6) == "-00042"

print("\nSign awareness assertions passed!")


## 3. The `rjust(width, '0')` Sign Bug vs. `zfill(width)`

Generic right-justification puts zeros to the left of the negative sign, resulting in corrupted numbers like `"000-42"`.
`zfill()` keeps the sign in front: `"-00042"`.

In [ ]:
num = "-50"

# Flawed: rjust() corrupts sign syntax
bad_rjust = num.rjust(6, "0")
print("rjust(6, '0'):", bad_rjust)  # '000-50'
assert bad_rjust == "000-50"

# Correct: zfill() preserves valid numeric format
good_zfill = num.zfill(6)
print("zfill(6):     ", good_zfill)  # '-00050'
assert good_zfill == "-00050"

print("\nrjust vs zfill sign contrast passed!")


## 4. Real-World Engineering Recipes

Explore three common software engineering patterns utilizing `str.zfill()`:
1. Standardized Enterprise Badge ID Generator
2. ISO 8601 Timestamp Formatter
3. Signed Telemetry / Financial Variance Formatter

In [ ]:
# Recipe 1: Badge generator
def make_badge(dept: str, seq: int) -> str:
    return f"{dept.upper()}-{str(seq).zfill(5)}"

assert make_badge("eng", 7) == "ENG-00007"
assert make_badge("sales", 14902) == "SALES-14902"

# Recipe 2: ISO timestamp generator
def iso_time(year: int, month: int, day: int, h: int, m: int, s: int) -> str:
    return f"{str(year).zfill(4)}-{str(month).zfill(2)}-{str(day).zfill(2)}T{str(h).zfill(2)}:{str(m).zfill(2)}:{str(s).zfill(2)}Z"

ts = iso_time(2026, 4, 9, 8, 5, 2)
print("Recipe 2 Timestamp:", ts)
assert ts == "2026-04-09T08:05:02Z"

# Recipe 3: Signed financial delta
def format_delta(delta: float, width: int = 8) -> str:
    prefix = "+" if delta > 0 else ""
    return f"{prefix}{delta:.2f}".zfill(width)

print("Recipe 3 Gain:", format_delta(14.50))
print("Recipe 3 Loss:", format_delta(-8.25))
assert format_delta(14.50) == "+0014.50"
assert format_delta(-8.25) == "-0008.25"

print("\nAll engineering recipes passed!")


## 5. Production Implementation & Self-Checking Unit Assertions

This production-grade `ZeroPadder` class encapsulates badge formatting, signed number padding, and ISO time unit formatting.

In [ ]:
class ZeroPadder:
    @staticmethod
    def format_badge(prefix: str, sequence_id: int, width: int = 6) -> str:
        """Creates uniform badge identifiers."""
        return f"{prefix}-{str(sequence_id).zfill(width)}"

    @staticmethod
    def pad_signed_value(value: int | float, total_width: int) -> str:
        """Pads signed numbers preserving positive/negative prefix."""
        s = f"+{value}" if value > 0 else str(value)
        return s.zfill(total_width)

    @staticmethod
    def pad_time_unit(unit: int) -> str:
        """Formats 2-digit date/time units."""
        return str(unit).zfill(2)


# ==========================================
# UNIT ASSERTION SUITE
# ==========================================
padder = ZeroPadder()

# 1. Basic zero padding
assert "42".zfill(5) == "00042"
assert "12345".zfill(3) == "12345"
assert "".zfill(4) == "0000"

# 2. Signed zero padding
assert "-42".zfill(6) == "-00042"
assert "+42".zfill(6) == "+00042"

# 3. Contrast with rjust
assert "-42".rjust(6, "0") == "000-42"

# 4. Badge formatting
assert padder.format_badge("USR", 125, 6) == "USR-000125"

# 5. Time unit padding
assert padder.pad_time_unit(5) == "05"
assert padder.pad_time_unit(12) == "12"

# 6. Signed value padding
assert padder.pad_signed_value(25, 6) == "+00025"
assert padder.pad_signed_value(-50, 6) == "-00050"

print("All str.zfill() production unit assertions passed successfully!")
